# FASE 09. Verificación e identificación open-set por huella de voz

Esta fase evalúa si los embeddings vocales permiten distinguir identidades y construye una base operacional de perfiles. Se mantienen dos análisis complementarios del notebook original:

1. **verificación pairwise**, que compara pares genuinos e impostores y calibra un umbral con identidades separadas de las de prueba;
2. **identificación open-set**, que decide si una consulta corresponde a un perfil conocido o debe rechazarse como identidad desconocida.

El notebook conserva visibles la restauración, preparación, filtros, construcción de muestras, splits sin fuga, calibración, evaluación, perfiles, ejemplos y visualizaciones. La lógica reusable permanece en `src/verificacion_huella_voz.py`.

## 09.0 — Configuración del entorno

**Kernel requerido:** `Python (TFM_HuellaDeVoz)`

In [ ]:
# INSTALACIÓN ACUMULADA DEL PROYECTO
from pathlib import Path

CURRENT_DIR = Path.cwd().resolve()
REPO_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
requirements_path = REPO_ROOT / "requirements.txt"

%pip install -q -r {requirements_path}

In [ ]:
# IMPORTS Y ACCESO AL REPOSITORIO
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from google.cloud import storage
from IPython.display import display

CURRENT_DIR = Path.cwd().resolve()
REPO_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src import verificacion_huella_voz as vp
from src.config import (
    DATA_DIR,
    EMBEDDING_VECTOR_CSV_DIR,
    GCS_UNAV_ROOT,
    GCS_VOICEPRINT_PREFIX,
    PROXY_GROUNDTRUTH_DIR,
    TRANSCRIPTION_ROOT,
    VOICEPRINT_DIR,
    ensure_phase09_directories,
)
from src.io_utils import read_csv_robust
from src.storage_io import download_directory, upload_directory

In [ ]:
# CONEXIÓN A GOOGLE CLOUD STORAGE
gcs_client = storage.Client()
print("Cliente GCS creado para el proyecto:", gcs_client.project)

In [ ]:
# CONFIGURACIÓN DE EJECUCIÓN
FORCE_RESTORE = False
FORCE_PREPARATION = False
FORCE_PAIRWISE = False
FORCE_OPEN_SET = False

ensure_phase09_directories()
vp.seed_everything()

print("Repositorio:", REPO_ROOT)
print("Outputs locales:", VOICEPRINT_DIR)
print("Prefijo GCS fase 09:", GCS_VOICEPRINT_PREFIX)
print("Forzar preparación:", FORCE_PREPARATION)
print("Forzar verificación pairwise:", FORCE_PAIRWISE)
print("Forzar identificación open-set:", FORCE_OPEN_SET)

## 09.1 — Restauración y decisión de reutilización

Primero se restaura el espacio completo de outputs de la fase 09. Si el manifiesto final y los artefactos operacionales están completos, no se recalculan los splits, pares, perfiles ni scores. Los inputs de embeddings, roles y transcripción solo se restauran cuando alguna parte debe reconstruirse.

In [ ]:
# RESTAURAR OUTPUTS PREVIOS DE LA FASE 09
restore_voiceprint = download_directory(
    local_dir=VOICEPRINT_DIR,
    gcs_prefix=GCS_UNAV_ROOT,
    gcs_client=gcs_client,
    base_dir=DATA_DIR,
)

force_any = FORCE_PREPARATION or FORCE_PAIRWISE or FORCE_OPEN_SET
phase09_reused = vp.voiceprint_outputs_complete() and not force_any

print("Restauración fase 09:", restore_voiceprint)
print("Ejecución final completa disponible:", vp.voiceprint_outputs_complete())
print("Se reutilizarán los resultados finales:", phase09_reused)

In [ ]:
# RESTAURAR INPUTS ÚNICAMENTE CUANDO HACE FALTA RECALCULAR
restore_embeddings = None
restore_proxy = None
restore_transcription = None

if not phase09_reused or FORCE_RESTORE:
    restore_embeddings = download_directory(
        local_dir=EMBEDDING_VECTOR_CSV_DIR,
        gcs_prefix=GCS_UNAV_ROOT,
        gcs_client=gcs_client,
        base_dir=DATA_DIR,
    )
    restore_proxy = download_directory(
        local_dir=PROXY_GROUNDTRUTH_DIR,
        gcs_prefix=GCS_UNAV_ROOT,
        gcs_client=gcs_client,
        base_dir=DATA_DIR,
    )
    restore_transcription = download_directory(
        local_dir=TRANSCRIPTION_ROOT,
        gcs_prefix=GCS_UNAV_ROOT,
        gcs_client=gcs_client,
        base_dir=DATA_DIR,
    )

print("Restauración embeddings:", restore_embeddings)
print("Restauración roles proxy:", restore_proxy)
print("Restauración transcripción:", restore_transcription)

## 09.2 — Inputs, claves de identidad y cobertura

Los embeddings segmentales proceden de la diarización y el reetiquetado. El mapping del Notebook 06 aporta el rol proxy y los identificadores anonimizados. La unión se realiza por audio y `speaker_final`; los valores `pd.NA` se conservan como ausentes y no se convierten en identidades de texto como `"nan"`.

In [ ]:
# CARGAR INPUTS O RESULTADOS PREPARADOS
if phase09_reused:
    restored_outputs = vp.load_voiceprint_outputs()
    df_voiceprint_segments = restored_outputs["segments"]
    df_samples = restored_outputs["samples"]
    identity_summary = restored_outputs["identity_summary"]
    df_identity_split = restored_outputs["identity_split"]
    emb_cols = vp.get_embedding_columns(df_samples)
    df_embeddings = pd.DataFrame()
    df_voice_segments = pd.DataFrame()
    coverage = pd.DataFrame()
    selected_columns = {}
    print("Preparación reutilizada desde los CSV de la fase 09.")
else:
    inputs = vp.load_voiceprint_inputs()
    df_embeddings = inputs["embeddings"]
    df_role_mapping = inputs["role_mapping"]
    df_segment_proxy = inputs["segment_proxy"]
    df_transcriptions = inputs["transcriptions"]
    emb_cols = inputs["embedding_columns"]

    print("Embeddings:", df_embeddings.shape)
    print("Mapping de roles:", df_role_mapping.shape)
    print("Proxy segmental opcional:", df_segment_proxy.shape)
    print("Transcripción opcional:", df_transcriptions.shape)
    print("Dimensión del embedding:", len(emb_cols))
    print("Fuente de transcripción:", inputs["transcription_path"])
    display(df_embeddings.head(3))
    display(df_role_mapping.head(3))

In [ ]:
# NORMALIZAR AUDIO, SPEAKER, ROL E IDENTIFICADORES
if not phase09_reused:
    (
        df_embeddings,
        df_role_mapping,
        df_role_mapping_small,
        selected_columns,
    ) = vp.normalize_voiceprint_inputs(df_embeddings, df_role_mapping)

    print("Columnas seleccionadas:", selected_columns)
    print("Mapping único audio-speaker:", len(df_role_mapping_small))
    display(df_role_mapping_small.head())
else:
    print("Normalización ya reflejada en los outputs restaurados.")

In [ ]:
# UNIR EMBEDDINGS CON ROLES PROXY Y AUDITAR COBERTURA
if not phase09_reused:
    df_voice_segments, coverage = vp.merge_embeddings_with_roles(
        df_embeddings,
        df_role_mapping_small,
        df_transcriptions,
    )
    print("Segmentos con embeddings:", len(df_embeddings))
    print(
        "Segmentos con rol AGENT/CLIENT:",
        int(df_voice_segments["role_proxy"].isin(["AGENT", "CLIENT"]).sum()),
    )
    print("Segmentos con person_id:", int(df_voice_segments["person_id"].notna().sum()))
    print(
        "Audios con identidad:",
        df_voice_segments.loc[df_voice_segments["person_id"].notna(), "audio_key"].nunique(),
    )
    display(coverage)
    display(df_voice_segments.head(3))
else:
    role_summary = (
        df_voiceprint_segments.groupby("role_proxy", dropna=False)
        .agg(
            n_segments=("person_id", "size"),
            n_audios=("audio_key", "nunique"),
            n_persons=("person_id", "nunique"),
        )
        .reset_index()
        if not df_voiceprint_segments.empty
        else pd.DataFrame()
    )
    display(role_summary)

## 09.3 — Filtros de calidad y muestras audio-persona

Se mantienen los filtros originales: duración entre 1,5 y 20 segundos, solapamiento máximo de 0,05, RMS mínimo de −40 dBFS y `valid_export` cuando existe. Después, cada identidad dentro de una llamada se resume mediante el centroide normalizado de sus segmentos.

In [ ]:
# FILTRAR SEGMENTOS CANDIDATOS PARA HUELLA
if not phase09_reused:
    cached_segments = vp.load_dataframe_checkpoint(
        vp.VOICEPRINT_SEGMENTS_CSV,
        force=FORCE_PREPARATION,
        required_columns=["audio_key", "person_id", "role_proxy"],
    )
    if cached_segments is None:
        df_voiceprint_segments = vp.filter_voiceprint_segments(
            df_voice_segments,
            emb_cols,
        )
        vp.save_dataframe_checkpoint(
            df_voiceprint_segments,
            vp.VOICEPRINT_SEGMENTS_CSV,
        )
        segment_status = "calculado"
    else:
        df_voiceprint_segments = cached_segments
        segment_status = "reutilizado"

print("Segmentos candidatos:", len(df_voiceprint_segments), "| estado:", segment_status if not phase09_reused else "restaurado")
if not df_voiceprint_segments.empty:
    segment_summary = (
        df_voiceprint_segments.groupby("role_proxy")
        .agg(
            n_segments=("person_id", "size"),
            n_audios=("audio_key", "nunique"),
            n_persons=("person_id", "nunique"),
            total_duration=("duration", "sum"),
        )
        .reset_index()
    )
    display(segment_summary)

In [ ]:
# CREAR MUESTRAS AUDIO-PERSONA
if not phase09_reused:
    cached_samples = vp.load_dataframe_checkpoint(
        vp.VOICEPRINT_SAMPLES_CSV,
        force=FORCE_PREPARATION,
        required_columns=["sample_id", "audio_key", "person_id", "role_proxy"],
    )
    if cached_samples is None:
        df_samples = vp.build_audio_person_samples(
            df_voiceprint_segments,
            emb_cols,
        )
        vp.save_dataframe_checkpoint(df_samples, vp.VOICEPRINT_SAMPLES_CSV)
        sample_status = "calculado"
    else:
        df_samples = cached_samples
        sample_status = "reutilizado"

print("Muestras audio-persona:", len(df_samples), "| estado:", sample_status if not phase09_reused else "restaurado")
print("Audios:", df_samples["audio_key"].nunique() if not df_samples.empty else 0)
print("Identidades:", df_samples["person_id"].nunique() if not df_samples.empty else 0)
if not df_samples.empty:
    sample_summary = (
        df_samples.groupby("role_proxy")
        .agg(
            n_samples=("sample_id", "size"),
            n_audios=("audio_key", "nunique"),
            n_persons=("person_id", "nunique"),
            mean_segments_per_sample=("n_segments", "mean"),
            mean_duration_sec=("sample_duration_sec", "mean"),
        )
        .reset_index()
    )
    display(sample_summary)

In [ ]:
# RESUMIR REPETICIÓN Y ELEGIBILIDAD DE IDENTIDADES
if not phase09_reused:
    cached_identity_summary = vp.load_dataframe_checkpoint(
        vp.VOICEPRINT_IDENTITY_SUMMARY_CSV,
        force=FORCE_PREPARATION,
        required_columns=["role_proxy", "person_id", "eligible_verification"],
    )
    if cached_identity_summary is None:
        identity_summary = vp.summarize_identities(df_samples)
        vp.save_dataframe_checkpoint(
            identity_summary,
            vp.VOICEPRINT_IDENTITY_SUMMARY_CSV,
        )
        identity_status = "calculado"
    else:
        identity_summary = cached_identity_summary
        identity_status = "reutilizado"

print("Identidades totales:", len(identity_summary), "| estado:", identity_status if not phase09_reused else "restaurado")
print(
    "Identidades elegibles para verificación:",
    int(identity_summary["eligible_verification"].astype(bool).sum()) if not identity_summary.empty else 0,
)
if not identity_summary.empty:
    display(
        identity_summary.groupby("role_proxy")
        .agg(
            n_persons=("person_id", "nunique"),
            n_eligible=("eligible_verification", "sum"),
            median_samples=("n_samples", "median"),
            max_samples=("n_samples", "max"),
        )
        .reset_index()
    )
    display(
        identity_summary.sort_values(
            ["eligible_verification", "n_samples", "total_duration_sec"],
            ascending=[False, False, False],
        ).head(20)
    )

## 09.4 — Verificación pairwise

La primera evaluación construye pares genuinos e impostores. El umbral se calibra con identidades de agentes y se aplica a agentes no vistos y clientes repetidos. El split se realiza por identidad, por lo que una misma persona no puede aparecer simultáneamente en calibración y test.

In [ ]:
# CREAR O REUTILIZAR SPLIT POR IDENTIDAD
if phase09_reused:
    pairwise_partition = vp.partition_from_pairwise_split(
        identity_summary,
        df_identity_split,
    )
else:
    cached_identity_split = vp.load_dataframe_checkpoint(
        vp.VOICEPRINT_IDENTITY_SPLIT_CSV,
        force=FORCE_PAIRWISE,
        required_columns=["person_id", "role_proxy", "split"],
    )
    if cached_identity_split is None:
        df_identity_split, pairwise_partition = vp.split_pairwise_identities(
            identity_summary
        )
        vp.save_dataframe_checkpoint(
            df_identity_split,
            vp.VOICEPRINT_IDENTITY_SPLIT_CSV,
        )
        split_status = "calculado"
    else:
        df_identity_split = cached_identity_split
        pairwise_partition = vp.partition_from_pairwise_split(
            identity_summary,
            df_identity_split,
        )
        split_status = "reutilizado"

calibration_ids = set(
    df_identity_split.loc[df_identity_split["split"].eq("calibration"), "person_id"].astype(str)
)
test_ids = set(
    df_identity_split.loc[df_identity_split["split"].ne("calibration"), "person_id"].astype(str)
)
print("Split:", split_status if not phase09_reused else "restaurado")
print("Agentes elegibles:", len(pairwise_partition["eligible_agents"]))
print("Clientes repetidos elegibles:", len(pairwise_partition["eligible_clients"]))
print("Identidades calibración:", len(calibration_ids))
print("Identidades test:", len(test_ids))
print("Intersección calibración/test:", len(calibration_ids.intersection(test_ids)))
display(df_identity_split.groupby(["role_proxy", "split"]).size().reset_index(name="n_identities"))

In [ ]:
# PREPARAR DATASETS Y GENERAR PARES
pairwise_sample_sets = vp.build_pairwise_sample_sets(
    df_samples,
    pairwise_partition,
)

pair_paths = {
    "calibration": vp.VOICEPRINT_PAIRS_CALIBRATION_CSV,
    "agent_test": vp.VOICEPRINT_PAIRS_AGENT_TEST_CSV,
    "client_test": vp.VOICEPRINT_PAIRS_CLIENT_TEST_CSV,
}
pair_seeds = {
    "calibration": vp.RANDOM_SEED,
    "agent_test": vp.RANDOM_SEED + 1,
    "client_test": vp.RANDOM_SEED + 2,
}

pairwise_pairs = {}
for dataset_name, sample_data in pairwise_sample_sets.items():
    cached_pairs = vp.load_dataframe_checkpoint(
        pair_paths[dataset_name],
        force=FORCE_PAIRWISE,
        required_columns=["same_identity", "similarity"],
    )
    if cached_pairs is None:
        cached_pairs = vp.build_verification_pairs(
            sample_data,
            emb_cols,
            random_state=pair_seeds[dataset_name],
        )
        vp.save_dataframe_checkpoint(cached_pairs, pair_paths[dataset_name])
    pairwise_pairs[dataset_name] = cached_pairs

print("Muestras calibración:", len(pairwise_sample_sets["calibration"]))
print("Muestras test agentes:", len(pairwise_sample_sets["agent_test"]))
print("Muestras test clientes:", len(pairwise_sample_sets["client_test"]))
print("Pares calibración:", len(pairwise_pairs["calibration"]))
print("Pares test agentes:", len(pairwise_pairs["agent_test"]))
print("Pares test clientes:", len(pairwise_pairs["client_test"]))
display(pairwise_pairs["calibration"].head())

In [ ]:
# CALIBRAR UMBRAL PAIRWISE
threshold_summary = vp.load_dataframe_checkpoint(
    vp.VOICEPRINT_PAIRWISE_THRESHOLD_CSV,
    force=FORCE_PAIRWISE,
    required_columns=["voiceprint_threshold", "threshold_strategy"],
)

if threshold_summary is None:
    calibration_pairs = pairwise_pairs["calibration"]
    if calibration_pairs.empty or calibration_pairs["same_identity"].nunique() < 2:
        raise ValueError(
            "No hay pares suficientes para calibrar el umbral pairwise."
        )
    pairwise_threshold_info = vp.choose_threshold_from_pairs(
        calibration_pairs,
        strategy=vp.THRESHOLD_STRATEGY,
    )
    VOICEPRINT_THRESHOLD = float(pairwise_threshold_info["threshold"])
    threshold_summary = pd.DataFrame([
        {
            "threshold_strategy": vp.THRESHOLD_STRATEGY,
            "voiceprint_threshold": VOICEPRINT_THRESHOLD,
            "eer_threshold": pairwise_threshold_info["eer_threshold"],
            "calibration_eer": pairwise_threshold_info["eer"],
            "n_calibration_pairs": len(calibration_pairs),
            "n_calibration_positive": int(calibration_pairs["same_identity"].eq(1).sum()),
            "n_calibration_negative": int(calibration_pairs["same_identity"].eq(0).sum()),
        }
    ])
    vp.save_dataframe_checkpoint(
        threshold_summary,
        vp.VOICEPRINT_PAIRWISE_THRESHOLD_CSV,
    )
else:
    VOICEPRINT_THRESHOLD = float(threshold_summary["voiceprint_threshold"].iloc[0])

print("Estrategia:", threshold_summary["threshold_strategy"].iloc[0])
print("Umbral calibrado:", round(VOICEPRINT_THRESHOLD, 4))
display(threshold_summary)

In [ ]:
# EVALUAR PAIRWISE Y CONSTRUIR MATRICES DE CONFUSIÓN
pairwise_metrics = vp.load_dataframe_checkpoint(
    vp.VOICEPRINT_PAIRWISE_METRICS_CSV,
    force=FORCE_PAIRWISE,
    required_columns=["dataset", "auc", "eer", "f1"],
)
if pairwise_metrics is None:
    pairwise_metrics = pd.DataFrame([
        vp.evaluate_pairs(
            pairwise_pairs["calibration"],
            VOICEPRINT_THRESHOLD,
            "calibration_agents",
        ),
        vp.evaluate_pairs(
            pairwise_pairs["agent_test"],
            VOICEPRINT_THRESHOLD,
            "test_agents_unseen",
        ),
        vp.evaluate_pairs(
            pairwise_pairs["client_test"],
            VOICEPRINT_THRESHOLD,
            "test_clients_repeated",
        ),
    ])
    vp.save_dataframe_checkpoint(
        pairwise_metrics,
        vp.VOICEPRINT_PAIRWISE_METRICS_CSV,
    )

df_confusion = vp.load_dataframe_checkpoint(
    vp.VOICEPRINT_PAIRWISE_CONFUSION_CSV,
    force=FORCE_PAIRWISE,
    required_columns=["dataset", "true_label"],
)
if df_confusion is None:
    df_confusion = pd.concat([
        vp.confusion_table(
            pairwise_pairs["calibration"],
            VOICEPRINT_THRESHOLD,
            "calibration_agents",
        ),
        vp.confusion_table(
            pairwise_pairs["agent_test"],
            VOICEPRINT_THRESHOLD,
            "test_agents_unseen",
        ),
        vp.confusion_table(
            pairwise_pairs["client_test"],
            VOICEPRINT_THRESHOLD,
            "test_clients_repeated",
        ),
    ], ignore_index=True)
    vp.save_dataframe_checkpoint(
        df_confusion,
        vp.VOICEPRINT_PAIRWISE_CONFUSION_CSV,
    )

display(pairwise_metrics)
display(df_confusion)

### Visualizaciones pairwise

Las distribuciones permiten comprobar si los pares de la misma identidad alcanzan similitudes mayores que los impostores. Las curvas ROC resumen la separación sin depender de un único umbral.

In [ ]:
# DISTRIBUCIONES DE SIMILITUD Y CURVAS ROC
pairwise_plot_specs = [
    (
        "calibration",
        "Distribución de similitud - calibración con agentes",
        vp.FIG_SIMILARITY_CALIBRATION,
        "Curva ROC - calibración con agentes",
        vp.FIG_ROC_CALIBRATION,
    ),
    (
        "agent_test",
        "Distribución de similitud - agentes no vistos",
        vp.FIG_SIMILARITY_AGENT_TEST,
        "Curva ROC - agentes no vistos",
        vp.FIG_ROC_AGENT_TEST,
    ),
    (
        "client_test",
        "Distribución de similitud - clientes repetidos",
        vp.FIG_SIMILARITY_CLIENT_TEST,
        "Curva ROC - clientes repetidos",
        vp.FIG_ROC_CLIENT_TEST,
    ),
]

for dataset_name, distribution_title, distribution_path, roc_title, roc_path in pairwise_plot_specs:
    pairs = pairwise_pairs[dataset_name]
    figure = vp.plot_similarity_distribution(
        pairs,
        distribution_title,
        threshold=VOICEPRINT_THRESHOLD,
    )
    if figure is not None:
        vp.save_figure(figure, distribution_path)
        display(figure)
        plt.close(figure)

    figure = vp.plot_roc_curve(pairs, roc_title)
    if figure is not None:
        vp.save_figure(figure, roc_path)
        display(figure)
        plt.close(figure)

identity_figure = vp.plot_identity_repetition(identity_summary)
if identity_figure is not None:
    vp.save_figure(identity_figure, vp.FIG_IDENTITY_REPETITION)
    display(identity_figure)
    plt.close(identity_figure)

In [ ]:
# EJEMPLOS DE PARES DIFÍCILES Y CLAROS
for dataset_name, title in [
    ("agent_test", "Agentes no vistos"),
    ("client_test", "Clientes repetidos"),
]:
    pairs = pairwise_pairs[dataset_name]
    print("=" * 80)
    print(title)
    if pairs.empty:
        print("No hay pares disponibles.")
        continue
    example_columns = [
        "same_identity",
        "similarity",
        "person_id_a",
        "person_id_b",
        "role_a",
        "role_b",
        "audio_a",
        "audio_b",
        "sample_id_a",
        "sample_id_b",
    ]
    display(pairs.sort_values("similarity", ascending=False)[example_columns].head(8))
    display(pairs.sort_values("similarity", ascending=True)[example_columns].head(8))

In [ ]:
# RESUMEN EJECUTIVO DE VERIFICACIÓN PAIRWISE
if phase09_reused:
    df_final_summary = restored_outputs["final_summary"]
else:
    df_final_summary = vp.build_pairwise_summary(
        embeddings_count=len(df_embeddings),
        candidates=df_voiceprint_segments,
        samples=df_samples,
        identity_summary=identity_summary,
        partition=pairwise_partition,
        threshold=VOICEPRINT_THRESHOLD,
        metrics=pairwise_metrics,
    )
    vp.save_dataframe_checkpoint(
        df_final_summary,
        vp.VOICEPRINT_FINAL_SUMMARY_CSV,
    )

display(df_final_summary)

## 09.5 — Identificación open-set

La evaluación open-set separa agentes conocidos para calibración, agentes conocidos no vistos durante la calibración y agentes completamente desconocidos que nunca forman perfiles. Cada identidad conocida aporta llamadas distintas para `enrollment` y `query`.

In [ ]:
# ELEGIBILIDAD PARA PERFILES OPEN-SET
if phase09_reused:
    df_samples_open_set = df_samples.copy()
    if "source_identity_id" not in df_samples_open_set.columns:
        df_samples_open_set["source_identity_id"] = df_samples_open_set["person_id"].astype("string")
    identity_summary_open_set = read_csv_robust(
        vp.VOICEPRINT_OPEN_SET_IDENTITY_SUMMARY_CSV
    )
else:
    cached_open_set_summary = vp.load_dataframe_checkpoint(
        vp.VOICEPRINT_OPEN_SET_IDENTITY_SUMMARY_CSV,
        force=FORCE_OPEN_SET,
        required_columns=["person_id", "role_proxy", "eligible_profile"],
    )
    df_samples_open_set, computed_open_set_summary = (
        vp.prepare_open_set_identity_summary(df_samples)
    )
    if cached_open_set_summary is None:
        identity_summary_open_set = computed_open_set_summary
        vp.save_dataframe_checkpoint(
            identity_summary_open_set,
            vp.VOICEPRINT_OPEN_SET_IDENTITY_SUMMARY_CSV,
        )
    else:
        identity_summary_open_set = cached_open_set_summary

open_set_eligibility = (
    identity_summary_open_set.groupby("role_proxy")
    .agg(
        identities=("person_id", "nunique"),
        eligible=("eligible_profile", "sum"),
        median_calls=("n_calls", "median"),
        max_calls=("n_calls", "max"),
    )
    .reset_index()
)
display(open_set_eligibility)

In [ ]:
# CREAR O REUTILIZAR SPLIT OPEN-SET
open_set_split = vp.load_dataframe_checkpoint(
    vp.VOICEPRINT_OPEN_SET_SPLIT_CSV,
    force=FORCE_OPEN_SET,
    required_columns=["sample_id", "identity_group", "sample_split"],
)
identity_partition = vp.load_json_checkpoint(
    vp.VOICEPRINT_IDENTITY_PARTITION_JSON,
    force=FORCE_OPEN_SET,
)

if open_set_split is None or identity_partition is None:
    eligible_open_set_agents = (
        identity_summary_open_set.loc[
            identity_summary_open_set["role_proxy"].eq("AGENT")
            & identity_summary_open_set["eligible_profile"].astype(bool),
            "person_id",
        ]
        .dropna()
        .astype(str)
        .tolist()
    )
    open_set_split, identity_partition = vp.build_open_set_split(
        df_samples_open_set,
        eligible_open_set_agents,
    )
    vp.save_dataframe_checkpoint(
        open_set_split,
        vp.VOICEPRINT_OPEN_SET_SPLIT_CSV,
    )
    vp.save_json_checkpoint(
        identity_partition,
        vp.VOICEPRINT_IDENTITY_PARTITION_JSON,
    )

print({key: len(value) for key, value in identity_partition.items()})
display(
    open_set_split.groupby(["identity_group", "sample_split"])
    .agg(
        identities=("person_id", "nunique"),
        samples=("sample_id", "size"),
        calls=("audio_key", "nunique"),
    )
    .reset_index()
)

In [ ]:
# CONSTRUIR PERFILES DE CALIBRACIÓN Y TEST
calibration_profiles = vp.load_dataframe_checkpoint(
    vp.VOICEPRINT_CALIBRATION_PROFILES_CSV,
    force=FORCE_OPEN_SET,
    required_columns=["profile_id", "source_identity_id"],
)
test_profiles = vp.load_dataframe_checkpoint(
    vp.VOICEPRINT_TEST_PROFILES_CSV,
    force=FORCE_OPEN_SET,
    required_columns=["profile_id", "source_identity_id"],
)

if calibration_profiles is None:
    calibration_enrollment = vp.get_samples_for_split(
        df_samples_open_set,
        open_set_split,
        "calibration_known",
        "enrollment",
    )
    calibration_profiles = vp.build_speaker_profiles(
        calibration_enrollment,
        emb_cols,
        "calibration",
    )
    vp.save_dataframe_checkpoint(
        calibration_profiles,
        vp.VOICEPRINT_CALIBRATION_PROFILES_CSV,
    )

if test_profiles is None:
    test_enrollment = vp.get_samples_for_split(
        df_samples_open_set,
        open_set_split,
        "test_known",
        "enrollment",
    )
    test_profiles = vp.build_speaker_profiles(
        test_enrollment,
        emb_cols,
        "test",
    )
    vp.save_dataframe_checkpoint(
        test_profiles,
        vp.VOICEPRINT_TEST_PROFILES_CSV,
    )

print("Perfiles calibración:", len(calibration_profiles))
print("Perfiles test:", len(test_profiles))
profile_columns = [
    "profile_id",
    "n_enrollment_calls",
    "n_enrollment_segments",
    "total_enrollment_duration_sec",
    "within_similarity_mean",
    "within_similarity_min",
]
display(calibration_profiles[profile_columns].head())

In [ ]:
# CALIBRAR UMBRAL OPEN-SET CON QUERIES NO UTILIZADAS EN ENROLLMENT
calibration_scores = vp.load_dataframe_checkpoint(
    vp.VOICEPRINT_CALIBRATION_SCORES_CSV,
    force=FORCE_OPEN_SET,
    required_columns=["same_identity", "similarity"],
)
calibration_predictions = vp.load_dataframe_checkpoint(
    vp.VOICEPRINT_CALIBRATION_PREDICTIONS_CSV,
    force=FORCE_OPEN_SET,
    required_columns=["best_profile_id", "best_similarity"],
)

if calibration_scores is None or calibration_predictions is None:
    calibration_queries = vp.get_samples_for_split(
        df_samples_open_set,
        open_set_split,
        "calibration_known",
        "query",
    )
    calibration_scores, calibration_predictions = vp.score_queries_against_profiles(
        calibration_queries,
        calibration_profiles,
        emb_cols,
        "calibration_known",
    )
    vp.save_dataframe_checkpoint(
        calibration_scores,
        vp.VOICEPRINT_CALIBRATION_SCORES_CSV,
    )
    vp.save_dataframe_checkpoint(
        calibration_predictions,
        vp.VOICEPRINT_CALIBRATION_PREDICTIONS_CSV,
    )

open_set_threshold_info = vp.load_json_checkpoint(
    vp.VOICEPRINT_THRESHOLDS_JSON,
    force=FORCE_OPEN_SET,
)
if open_set_threshold_info is None:
    open_set_threshold_info = vp.choose_verification_threshold(
        calibration_scores,
        vp.OPEN_SET_THRESHOLD_STRATEGY,
    )
    open_set_threshold_info.update({
        "created_at_utc": vp.utc_now_iso(),
        "embedding_model": vp.EMBEDDING_MODEL_LABEL,
        "embedding_dim": len(emb_cols),
        "notebook_version": vp.NOTEBOOK_VERSION,
        "calibration_identity_count": int(
            calibration_profiles["profile_id"].nunique()
        ),
    })
    vp.save_json_checkpoint(
        open_set_threshold_info,
        vp.VOICEPRINT_THRESHOLDS_JSON,
    )

OPEN_SET_THRESHOLD = float(open_set_threshold_info["acceptance_threshold"])
print("Umbral open-set:", round(OPEN_SET_THRESHOLD, 4))
print("Estrategia:", open_set_threshold_info["strategy"])
print("EER calibración:", round(float(open_set_threshold_info["eer"]), 4))
print("AUC calibración:", round(float(open_set_threshold_info["calibration_auc"]), 4))

In [ ]:
# EVALUAR QUERIES CONOCIDAS Y DESCONOCIDAS
score_specs = [
    (
        "test_known",
        vp.VOICEPRINT_TEST_KNOWN_SCORES_CSV,
        vp.VOICEPRINT_TEST_KNOWN_PREDICTIONS_CSV,
        test_profiles,
    ),
    (
        "test_unknown",
        vp.VOICEPRINT_TEST_UNKNOWN_SCORES_CSV,
        vp.VOICEPRINT_TEST_UNKNOWN_PREDICTIONS_CSV,
        test_profiles,
    ),
]

open_set_scores = {}
open_set_top1_predictions = {}
for group_name, scores_path, predictions_path, profiles in score_specs:
    scores = vp.load_dataframe_checkpoint(
        scores_path,
        force=FORCE_OPEN_SET,
        required_columns=["similarity", "same_identity"],
    )
    predictions = vp.load_dataframe_checkpoint(
        predictions_path,
        force=FORCE_OPEN_SET,
        required_columns=["best_similarity", "best_profile_id"],
    )
    if scores is None or predictions is None:
        queries = vp.get_samples_for_split(
            df_samples_open_set,
            open_set_split,
            group_name,
            "query",
        )
        scores, predictions = vp.score_queries_against_profiles(
            queries,
            profiles,
            emb_cols,
            group_name,
        )
        vp.save_dataframe_checkpoint(scores, scores_path)
        vp.save_dataframe_checkpoint(predictions, predictions_path)
    open_set_scores[group_name] = scores
    open_set_top1_predictions[group_name] = predictions

cached_open_set_predictions = vp.load_dataframe_checkpoint(
    vp.VOICEPRINT_OPEN_SET_PREDICTIONS_CSV,
    force=FORCE_OPEN_SET,
    required_columns=["decision", "identification_correct"],
)
if cached_open_set_predictions is None:
    df_open_set_predictions = pd.concat([
        vp.apply_open_set_decision(
            open_set_top1_predictions["test_known"],
            OPEN_SET_THRESHOLD,
        ),
        vp.apply_open_set_decision(
            open_set_top1_predictions["test_unknown"],
            OPEN_SET_THRESHOLD,
        ),
    ], ignore_index=True)
    vp.save_dataframe_checkpoint(
        df_open_set_predictions,
        vp.VOICEPRINT_OPEN_SET_PREDICTIONS_CSV,
    )
else:
    df_open_set_predictions = cached_open_set_predictions

prediction_columns = [
    "query_group",
    "audio_key",
    "true_source_identity_id",
    "best_profile_id",
    "best_source_identity_id",
    "best_similarity",
    "top1_top2_margin",
    "decision",
    "identification_correct",
]
display(df_open_set_predictions[prediction_columns].head(20))

In [ ]:
# MÉTRICAS Y MATRIZ KNOWN / UNKNOWN
identification_metrics = vp.load_json_checkpoint(
    vp.VOICEPRINT_IDENTIFICATION_METRICS_JSON,
    force=FORCE_OPEN_SET,
)
verification_metrics = vp.load_dataframe_checkpoint(
    vp.VOICEPRINT_VERIFICATION_METRICS_CSV,
    force=FORCE_OPEN_SET,
    required_columns=["dataset", "auc", "eer"],
)

if identification_metrics is None or verification_metrics is None:
    (
        identification_metrics,
        verification_metrics,
        predictions_by_truth,
    ) = vp.evaluate_open_set_predictions(
        df_open_set_predictions,
        calibration_scores,
        open_set_scores["test_known"],
        OPEN_SET_THRESHOLD,
    )
    vp.save_json_checkpoint(
        identification_metrics,
        vp.VOICEPRINT_IDENTIFICATION_METRICS_JSON,
    )
    vp.save_dataframe_checkpoint(
        verification_metrics,
        vp.VOICEPRINT_VERIFICATION_METRICS_CSV,
    )

df_decision_matrix = vp.load_dataframe_checkpoint(
    vp.VOICEPRINT_OPEN_SET_CONFUSION_CSV,
    force=FORCE_OPEN_SET,
    required_columns=["true_class", "pred_known", "pred_unknown"],
)
if df_decision_matrix is None:
    df_decision_matrix = vp.build_open_set_confusion(
        df_open_set_predictions
    )
    vp.save_dataframe_checkpoint(
        df_decision_matrix,
        vp.VOICEPRINT_OPEN_SET_CONFUSION_CSV,
    )

display(verification_metrics)
display(pd.DataFrame([identification_metrics]))
display(df_decision_matrix)

### Visualizaciones open-set

La primera figura muestra el umbral sobre las comparaciones genuinas e impostoras de calibración. La segunda compara la mejor similitud encontrada por queries conocidas y desconocidas; es la visualización más directa para explicar la lógica de rechazo open-set.

In [ ]:
# DISTRIBUCIÓN DE CALIBRACIÓN, KNOWN VS UNKNOWN Y ROC
calibration_figure = vp.plot_similarity_distribution(
    calibration_scores,
    "Calibración de huella de voz",
    threshold=OPEN_SET_THRESHOLD,
)
if calibration_figure is not None:
    vp.save_figure(calibration_figure, vp.FIG_OPEN_SET_CALIBRATION)
    display(calibration_figure)
    plt.close(calibration_figure)

known_unknown_figure = vp.plot_open_set_known_unknown(
    df_open_set_predictions,
    OPEN_SET_THRESHOLD,
)
if known_unknown_figure is not None:
    vp.save_figure(known_unknown_figure, vp.FIG_OPEN_SET_KNOWN_UNKNOWN)
    display(known_unknown_figure)
    plt.close(known_unknown_figure)

open_set_roc_figure = vp.plot_roc_curve(
    calibration_scores,
    "ROC de verificación vocal",
)
if open_set_roc_figure is not None:
    vp.save_figure(open_set_roc_figure, vp.FIG_OPEN_SET_ROC)
    display(open_set_roc_figure)
    plt.close(open_set_roc_figure)

In [ ]:
# CONSISTENCIA INTERNA DE LOS PERFILES
profile_consistency_columns = [
    "profile_id",
    "n_enrollment_calls",
    "total_enrollment_duration_sec",
    "within_similarity_mean",
    "within_similarity_min",
]
profile_consistency = pd.concat([
    calibration_profiles[profile_consistency_columns].assign(profile_set="calibration"),
    test_profiles[profile_consistency_columns].assign(profile_set="test"),
], ignore_index=True)

display(
    profile_consistency.sort_values(
        "within_similarity_min",
        ascending=True,
    ).head(15)
)

if not profile_consistency.empty:
    figure, axis = plt.subplots(figsize=(7, 5))
    for profile_set, group in profile_consistency.groupby("profile_set"):
        axis.scatter(
            group["total_enrollment_duration_sec"],
            group["within_similarity_min"],
            alpha=0.7,
            label=profile_set,
        )
    axis.set_title("Consistencia mínima de perfiles según audio de enrollment")
    axis.set_xlabel("Duración total de enrollment, segundos")
    axis.set_ylabel("Similitud mínima con el centroide")
    axis.legend()
    axis.grid(True, alpha=0.3)
    display(figure)
    plt.close(figure)

## 09.6 — Base operacional y contrato de inferencia

Tras la evaluación, se construyen perfiles con todas las llamadas elegibles disponibles. La salida de una consulta puede ser una identidad conocida o `UNKNOWN`; una identidad rechazada no se incorpora automáticamente a la base oficial.

In [ ]:
# CONSTRUIR PERFILES OPERACIONALES
agent_profiles_operational = vp.load_dataframe_checkpoint(
    vp.VOICEPRINT_AGENT_PROFILES_CSV,
    force=FORCE_OPEN_SET,
    required_columns=["profile_id", "source_identity_id"],
)
if agent_profiles_operational is None:
    agent_profiles_operational = vp.build_operational_profiles(
        df_samples_open_set,
        identity_summary_open_set,
        emb_cols,
        role="AGENT",
    )
    vp.save_dataframe_checkpoint(
        agent_profiles_operational,
        vp.VOICEPRINT_AGENT_PROFILES_CSV,
    )

client_profiles_operational = pd.DataFrame()
if vp.BUILD_CLIENT_PROFILES:
    client_profiles_operational = vp.load_dataframe_checkpoint(
        vp.VOICEPRINT_CLIENT_PROFILES_CSV,
        force=FORCE_OPEN_SET,
        required_columns=["profile_id", "source_identity_id"],
    )
    if client_profiles_operational is None:
        client_profiles_operational = vp.build_operational_profiles(
            df_samples_open_set,
            identity_summary_open_set,
            emb_cols,
            role="CLIENT",
        )
        vp.save_dataframe_checkpoint(
            client_profiles_operational,
            vp.VOICEPRINT_CLIENT_PROFILES_CSV,
        )

print("Perfiles operacionales de agentes:", len(agent_profiles_operational))
print("Perfiles operacionales de clientes:", len(client_profiles_operational))

In [ ]:
# GUARDAR METADATA Y PROBAR EL CONTRATO DE INFERENCIA
model_metadata = vp.load_json_checkpoint(
    vp.VOICEPRINT_MODEL_METADATA_JSON,
    force=FORCE_OPEN_SET,
)
if model_metadata is None:
    model_metadata = vp.build_model_metadata(
        emb_cols,
        open_set_threshold_info,
        agent_profiles_operational,
        client_profiles_operational,
    )
    vp.save_json_checkpoint(
        model_metadata,
        vp.VOICEPRINT_MODEL_METADATA_JSON,
    )

display(pd.DataFrame([{
    "config_hash": model_metadata["config_hash"],
    "embedding_dim": model_metadata["embedding_dim"],
    "threshold": model_metadata["acceptance_threshold"],
    "agent_profiles": model_metadata["n_agent_profiles"],
    "client_profiles": model_metadata["n_client_profiles"],
}]))

demo_query = df_samples_open_set[
    df_samples_open_set["role_proxy"].eq("AGENT")
].head(1).copy()
if not demo_query.empty:
    demo_result = vp.identify_query_samples(demo_query)
    display(demo_result[[
        "sample_id",
        "best_profile_id",
        "best_source_identity_id",
        "best_similarity",
        "top1_top2_margin",
        "decision",
    ]])

## 09.7 — Resumen final, manifiesto y sincronización

El manifiesto se escribe únicamente después de comprobar los artefactos operacionales obligatorios. La carpeta completa se sincroniza al prefijo exclusivo de la fase 09 preservando checkpoints, figuras, CSV y JSON.

In [ ]:
# RESUMEN FINAL OPEN-SET Y MANIFIESTO
if phase09_reused:
    df_open_set_final_summary = restored_outputs["open_set_summary"]
    run_manifest = restored_outputs["manifest"]
else:
    df_open_set_final_summary = vp.build_open_set_final_summary(
        df_voiceprint_segments,
        df_samples_open_set,
        identity_summary_open_set,
        open_set_threshold_info,
        identification_metrics,
        agent_profiles_operational,
        client_profiles_operational,
    )
    vp.save_dataframe_checkpoint(
        df_open_set_final_summary,
        vp.VOICEPRINT_OPEN_SET_SUMMARY_CSV,
    )
    run_manifest = vp.build_run_manifest(
        model_metadata,
        df_open_set_final_summary,
    )
    vp.save_json_checkpoint(
        run_manifest,
        vp.VOICEPRINT_SUCCESS_JSON,
    )

display(df_open_set_final_summary)
print("Estado del manifiesto:", run_manifest.get("status"))
print("Outputs faltantes:", run_manifest.get("missing_outputs", []))

In [ ]:
# AUDITORÍA DE OUTPUTS PRINCIPALES
output_paths = vp.PAIRWISE_OUTPUTS + vp.OPEN_SET_REQUIRED_OUTPUTS
output_audit = pd.DataFrame([
    {
        "file": path.name,
        "relative_path": path.relative_to(VOICEPRINT_DIR).as_posix(),
        "exists": path.exists(),
        "size_bytes": path.stat().st_size if path.exists() else 0,
    }
    for path in output_paths
])

display(output_audit)
missing_outputs = output_audit.loc[~output_audit["exists"], "relative_path"].tolist()
if missing_outputs:
    raise FileNotFoundError(
        "Faltan outputs finales de la fase 09: " + ", ".join(missing_outputs)
    )

In [ ]:
# SINCRONIZAR LA FASE 09 CON SU PREFIJO GCS
upload_summary = upload_directory(
    local_dir=VOICEPRINT_DIR,
    gcs_prefix=GCS_VOICEPRINT_PREFIX,
    gcs_client=gcs_client,
    skip_unchanged=True,
)

print("Resumen de subida:", upload_summary)
print("Destino:", GCS_VOICEPRINT_PREFIX)

In [ ]:
# CONTROL FINAL
print("FASE 09 COMPLETADA")
print("Segmentos candidatos:", len(df_voiceprint_segments))
print("Muestras audio-persona:", len(df_samples_open_set))
print("Perfiles operacionales de agentes:", len(agent_profiles_operational))
print("Perfiles operacionales de clientes:", len(client_profiles_operational))
print("Umbral open-set:", round(OPEN_SET_THRESHOLD, 4))
print("Manifiesto:", vp.VOICEPRINT_SUCCESS_JSON)
print("Outputs locales:", VOICEPRINT_DIR)
print("Outputs GCS:", GCS_VOICEPRINT_PREFIX)